# 07 — CI Automated Evaluation (GitHub Action)

**What you'll learn:**
- How the [GenAI Evaluation GitHub Action](https://github.com/marketplace/actions/generative-ai-evaluation-action)
  runs `azure-ai-evaluation` in CI
- The three pieces: config JSON, data JSONL, workflow YAML
- OIDC auth and cost-control triggers

> **This notebook is documentation-only.** The files it describes live in
> [`../ci-example/`](../ci-example/). They sit *outside* `.github/workflows/` on purpose,
> so nothing runs automatically in this repo. Copy them into your own repo to use them.

---

## Why gate CI on evaluation?

A prompt tweak that improves one answer can silently regress ten others. Wiring
evaluation into CI turns "looks good to me" into an automated quality gate: the Action
runs your evaluators on a test dataset and writes scores into the **Actions job summary**
on every triggered run.

## Piece 1 — the config (`evaluate-config.json`)

The Action reads a JSON config pointing at your data, the evaluators to run, and the
judge model. See [`../ci-example/evaluate-config.json`](../ci-example/evaluate-config.json):

```json
{
  "data": "${{ github.workspace }}/evaluate-with-foundry-evals/ci-example/eval-input.jsonl",
  "evaluators": {
    "coherence": "CoherenceEvaluator",
    "fluency": "FluencyEvaluator",
    "relevance": "RelevanceEvaluator",
    "f1": "F1ScoreEvaluator"
  },
  "ai_model_configuration": {
    "type": "azure_openai",
    "azure_endpoint": "${AZURE_OPENAI_ENDPOINT}",
    "azure_deployment": "${AZURE_OPENAI_CHAT_DEPLOYMENT}",
    "api_key": "${AZURE_OPENAI_API_KEY}",
    "api_version": "${AZURE_OPENAI_API_VERSION}"
  }
}
```

Evaluator names map to the same classes you used in notebooks 01–02.

## Piece 2 — the data (`eval-input.jsonl`)

Same JSONL schema as the notebooks — `query`, `response`, `ground_truth`, `context`.
See [`../ci-example/eval-input.jsonl`](../ci-example/eval-input.jsonl):

```json
{"query": "What is a deductible?", "response": "A deductible is what you pay before insurance pays the rest.", "ground_truth": "The amount the policyholder pays before the insurer covers the rest.", "context": "Glossary: Deductible - the portion a policyholder pays first."}
```

In a real repo this is your curated regression set — grow it whenever you find a bug.

## Piece 3 — the workflow (`evaluate.yml`)

See [`../ci-example/evaluate.yml`](../ci-example/evaluate.yml). Key parts:

```yaml
on:
  pull_request:
    paths:
      - "prompts/**"            # only run when prompts/data change — control cost
  workflow_dispatch:

permissions:
  id-token: write               # OIDC — no stored credentials
  contents: read

jobs:
  evaluate:
    runs-on: ubuntu-latest
    env:
      GENAI_EVALS_CONFIG_PATH: ${{ github.workspace }}/.../evaluate-config.json
    steps:
      - uses: actions/checkout@v4
      - uses: azure/login@v2      # federated OIDC login to Azure
        with:
          client-id: ${{ secrets.OIDC_AZURE_CLIENT_ID }}
          tenant-id: ${{ secrets.OIDC_AZURE_TENANT_ID }}
          subscription-id: ${{ secrets.OIDC_AZURE_SUBSCRIPTION_ID }}
      - uses: microsoft/genai-evals@main
        with:
          evaluate-configuration: ${{ env.GENAI_EVALS_CONFIG_PATH }}
```

## Auth & secrets

The Action authenticates to Azure with **OIDC** (`azure/login@v2`) — no long-lived keys
in the repo. Configure a federated credential on an Entra app registration, then set
these repo/environment secrets:

| Secret | Purpose |
|--------|---------|
| `OIDC_AZURE_CLIENT_ID` / `TENANT_ID` / `SUBSCRIPTION_ID` | federated login |
| `AZURE_OPENAI_ENDPOINT` | judge model endpoint |
| `AZURE_OPENAI_CHAT_DEPLOYMENT` | judge deployment name |
| `AZURE_OPENAI_API_VERSION` | API version |
| `AZURE_OPENAI_API_KEY` | judge key (only if not using Entra ID for the judge) |

## Cost control

Judge-model evaluators cost API calls, so **don't run on every commit**:

- Trigger on `pull_request` with a `paths:` filter (prompts, eval data) — not every push.
- Keep the CI dataset small and focused; run the full suite on a schedule or on demand.
- Prefer cheap NLP evaluators (F1) for fast gates; reserve AI-assisted metrics for PRs.

## Reading results

After a run, open **Actions → the workflow run → job summary**. The Action writes a
metrics table there. Fail the job on a threshold (e.g. mean relevance < 4) to *block*
regressions, or keep it advisory while you build confidence.

## Recap

- The Action runs `azure-ai-evaluation` in CI from a config + data file.
- OIDC (`azure/login`) keeps credentials out of the repo.
- Gate on `paths:` and small datasets to control cost; results land in the job summary.
- Working reference files: [`../ci-example/`](../ci-example/).

**Done!** Back to the [workshop README](../README.md).